# Neural Networks Model

In [15]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import random
import numpy as np
import pandas as pd

import copy

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix

import seaborn as sns
import matplotlib.pylab as plt
%matplotlib inline

from data_cleaning import cleaned_data

random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)

In [16]:
class Dataset:
    """
    Prepare the dataset into training, validation, and test sets.
    """
    def __init__(self, random_state = 123):
        self.data = cleaned_data()
        y_df = self.data["VOTED"].map({"voted": 0, "not_voted": 1}).to_numpy()

        categorical_feature = ["SEX", "RACE", "EDUC", "EMPSTAT", "NATIVITY", "REGION",
                               "METRO", "MARST", "DIFFMOB"]
        numerical_feature = ["AGE", "FAMSIZE", "NCHILD", "FAMINC", "INCOME_PER_PERSON"]

        ## training set 70%, validation set 10%, test set 20%
        train_x, rest_x, self.train_y, rest_y = train_test_split(
            self.data, y_df, test_size=0.3, random_state=random_state)
        val_x, test_x, self.val_y, self.test_y = train_test_split(
            rest_x, rest_y, test_size=(2/3), random_state=random_state)
        
        self.ohe = OneHotEncoder(sparse_output=False)
        self.scaler = StandardScaler()

        train_x_categorical = self.ohe.fit_transform(train_x[categorical_feature])
        train_x_numerical = self.scaler.fit_transform(train_x[numerical_feature])
        val_x_categorical = self.ohe.transform(val_x[categorical_feature])
        val_x_numerical = self.scaler.transform(val_x[numerical_feature])
        test_x_categorical = self.ohe.transform(test_x[categorical_feature])
        test_x_numerical = self.scaler.transform(test_x[numerical_feature])

        self.train_x = np.concatenate([train_x_categorical, train_x_numerical], axis=1)
        self.val_x = np.concatenate([val_x_categorical, val_x_numerical], axis=1)
        self.test_x = np.concatenate([test_x_categorical, test_x_numerical], axis=1)

        categorical_feature_name = self.ohe.get_feature_names_out(categorical_feature)
        numerical_feature_name = self.scaler.get_feature_names_out(numerical_feature)
        self.feature_order = np.concatenate([categorical_feature_name, numerical_feature_name])

In [17]:
dataset_handler = Dataset(random_state=1234)

print("\n"*2, dataset_handler.train_x.shape)
print("\n", dataset_handler.val_x.shape)
print("\n", dataset_handler.test_x.shape, "\n"*2)
print(dataset_handler.feature_order, "\n"*2)
print(dataset_handler.train_x[:3])



 (43628, 31)

 (6233, 31)

 (12466, 31) 


['SEX_female' 'SEX_male' 'RACE_asian' 'RACE_black'
 'RACE_indian_aleut_eskimo' 'RACE_others' 'RACE_white' 'EDUC_college_grad'
 'EDUC_hs_grad' 'EDUC_master_higher' 'EMPSTAT_employed'
 'EMPSTAT_not_in_labor_force' 'EMPSTAT_retired' 'EMPSTAT_unemployed'
 'NATIVITY_foreign_born' 'NATIVITY_native_born' 'REGION_midwest'
 'REGION_northeast' 'REGION_south' 'REGION_west' 'METRO_metropolitan'
 'METRO_not_metropolitan' 'MARST_has_spouse' 'MARST_no_spouse'
 'DIFFMOB_mobility_limitation' 'DIFFMOB_no_mobility_limitation' 'AGE'
 'FAMSIZE' 'NCHILD' 'FAMINC' 'INCOME_PER_PERSON'] 


[[ 1.          0.          0.          0.          0.          0.
   1.          1.          0.          0.          0.          1.
   0.          0.          1.          0.          1.          0.
   0.          0.          1.          0.          0.          1.
   0.          1.         -0.43837777  2.28542562 -0.58485114 -1.53299425
  -1.33195994]
 [ 0.          1.          0. 

In [18]:
class TorchDataset():

    def __init__(self, dataset_handler, batch_size):
        self.X_train = torch.from_numpy(dataset_handler.train_x).float()
        self.y_train = torch.from_numpy(dataset_handler.train_y).float().view(-1, 1)

        self.X_val = torch.from_numpy(dataset_handler.val_x).float()
        self.y_val = torch.from_numpy(dataset_handler.val_y).float().view(-1, 1)

        self.X_test = torch.from_numpy(dataset_handler.test_x).float()
        self.y_test = torch.from_numpy(dataset_handler.test_y).float().view(-1, 1)

        ## Pure SGD takes too long to train the model. Therefore, mini-batch training is introduced.
        self.train_loader = DataLoader(TensorDataset(self.X_train, self.y_train), batch_size=batch_size, shuffle=True)

In [19]:
class TorchNetwork(nn.Module):

    def __init__(self, num_features, hidden_layer, hidden_dim):
        super().__init__()
        layers = []
        ## Input Layer
        layers.append(nn.Linear(num_features, hidden_dim))
        layers.append(nn.ReLU())
        for _ in range(hidden_layer-1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, 1))
        self.n = nn.Sequential(*layers)

    def forward(self, x):
        return self.n(x)

class Model():
    
    def __init__(self, dataset, batch_size, hidden_layer, hidden_dim, lr, penalty):
        self.torch_dataset = TorchDataset(dataset, batch_size)
        self.model = TorchNetwork(self.torch_dataset.X_train.shape[1], hidden_layer, hidden_dim)
        self.optim = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=penalty)
        self.pos_weight = torch.tensor([(self.torch_dataset.y_train == 0).sum()/(self.torch_dataset.y_train == 1).sum()]).float()
        self.loss_func = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)

    def train(self, epochs=50, threshold=0.5):

        f1_history = []
        initial_loss = None

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0

            for x, y in self.torch_dataset.train_loader:
                self.optim.zero_grad()
            
                output = self.model(x)
                batch_loss = self.loss_func(output, y)

                batch_loss.backward()
                self.optim.step()
                total_loss += batch_loss.item() * x.size(0)

            avg_loss = total_loss/len(self.torch_dataset.X_train)

            if epoch == 0 :
                initial_loss = avg_loss
            elif epoch > 5 and avg_loss > initial_loss:
                return []
    
            self.model.eval()
            with torch.no_grad():
                result = self.model(self.torch_dataset.X_val)
                hat_y = (torch.sigmoid(result) > threshold).float()
                f1 = f1_score(self.torch_dataset.y_val.numpy(), hat_y.numpy())
                f1_history.append(f1)

        return f1_history

    def test(self, threshold=0.5):
        self.model.eval()
        with torch.no_grad():
            result = self.model(self.torch_dataset.X_test)
            hat_y = (torch.sigmoid(result) > threshold).float()
            f1 = f1_score(self.torch_dataset.y_test.numpy(), hat_y.numpy())
            conf_matrix = confusion_matrix(self.torch_dataset.y_test.numpy(), hat_y.numpy())
            tn, fp, fn, tp = conf_matrix[0,0], conf_matrix[0,1], conf_matrix[1,0], conf_matrix[1,1]
        print("F1:", f1, "\n")
        print("False negative rate:", fn / (fn + tp), "\n")
        print("False positive rate:", fp / (fp + tn), "\n")


In [20]:
batch_size = [50, 100, 200]
hidden_layer = [2, 5, 10]
hidden_dim = [31, 62]
learning_rate = [0.01, 0.001, 0.0001]
penalty = [0.01, 0.001, 0.0001]

best_f1_mean = 0
best_hyperparameter = None
best_model = None
results = []

for bs in batch_size:
    for hl in hidden_layer:
        for hd in hidden_dim:
            for lr in learning_rate:
                for p in penalty:
                    model = Model(dataset_handler, bs, hl, hd, lr, p)
                    f1_history = model.train(epochs=20)
                    result = {"batch_size": bs, "hidden_layer": hl,
                              "hidden_dim": hd, "learning_rate": lr,
                              "penalty": p}

                    if not f1_history or len(f1_history) < 5:
                        result["f1"] = 0
                        results.append(result)
                        continue
                    f1_std = np.std(f1_history)
                    if f1_std > 0.2 or f1_std < 0.001:
                        result["f1"] = 0
                        results.append(result)
                        continue
                    current_f1_mean = np.mean(f1_history[-5:])
                    result["f1"] = current_f1_mean
                    results.append(result)
                    if current_f1_mean > best_f1_mean:
                        best_f1_mean = current_f1_mean
                        best_hyperparameter = {"Batch size": bs, "Number of hidden layers": hl,
                                               "Hidden dimension": hd, "Learning rate": lr, "Penalty rate": p}
                        best_model = copy.deepcopy(model.model.state_dict())
                        
print("Best F1:", best_f1_mean)
print("Best Model", best_hyperparameter)

Best F1: 0.520361267996705
Best Model {'Batch size': 100, 'Number of hidden layers': 2, 'Hidden dimension': 62, 'Learning rate': 0.0001, 'Penalty rate': 0.001}


In [30]:
final_test_model = Model(dataset_handler, best_hyperparameter["Batch size"],
                         best_hyperparameter["Number of hidden layers"],
                         best_hyperparameter["Hidden dimension"],
                         best_hyperparameter["Learning rate"],
                         best_hyperparameter["Penalty rate"])

final_test_model.model.load_state_dict(best_model)
final_test_model.test()

F1: 0.5113692535837865 

False negative rate: 0.28875902371949125 

False positive rate: 0.32583446688291307 

